# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is described by a Croissant schema and is loaded directly from a remote JSON-LD file referenced below.

In [ ]:
# Ensure mlcroissant is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset and inspect main metadata attributes
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print high-level dataset information
print(f"Name: {metadata.name}\n")
print("Description:")
print(metadata.description)


## 2. Data Overview
Review available record sets, fields, and their IDs.

**Tip:** All entity references use their `@id` fields, as defined in the schema, for consistent querying.

In [ ]:
# List available record sets and their fields
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in metadata!\nInspecting alternate structure (such as distribution, hasPart, or files)...\n")
else:
    for record_set in record_sets:
        print(f"Record set: {record_set['@id']} - {record_set.get('name', 'N/A')}")
        if 'field' in record_set:
            fields = record_set['field']
            if not isinstance(fields, list):
                fields = [fields]
            for field in fields:
                if isinstance(field, dict):
                    print(f"  Field: {field.get('@id', 'N/A')} ({field.get('name','N/A')})")
                else:
                    print(f"  Field: {field}")
else:
    # If no record_sets found, attempt to peek at dataset.records() directly for likely @id.
    print("Attempting to list records set by peeking at records() generator.")


## 3. Data Extraction
Load data from the main record set(s) into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we extract all defined record sets. If multiple record sets are present, all will be loaded into separate DataFrames for further processing.

In [ ]:
# Extract data from each record set

# In this dataset, the main record set, as defined by schema convention, is typically 'cr:RecordSet' or found by listing dataset.record_sets
record_set_ids = [rs["@id"] for rs in dataset.record_sets]
if not record_set_ids:
    # In some datasets, there may be only one main record set whose @id follows the file path or name
    # Try to guess by extracting via records()
    try:
        # The mlcroissant library may allow records() with default record set
        try_records = list(dataset.records())
        print(f"Default records loaded, showing columns: {list(try_records[0].keys()) if try_records else 'No records!'}")
        df = pd.DataFrame(try_records)
        display(df.head())
        # Save for later use
        dataframes = { 'main': df }
        main_record_set_id = 'main'
    except Exception as e:
        print("Could not extract default records automatically: ", str(e))
        dataframes = {}
        main_record_set_id = None
else:
    dataframes = {}
    for record_set_id in record_set_ids:
        print(f"Loading records for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
    main_record_set_id = record_set_ids[0] if record_set_ids else None

if main_record_set_id:
    print("\nColumns in main record set:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head(5))
else:
    print("No record sets were loaded.")

## 4. Exploratory Data Analysis (EDA)
We now explore and process the dataset. Operations include:
- Filtering records by a numeric field,
- Normalizing values,
- Optionally grouping by a categorical field.

All fields are referenced by their schema `@id` (i.e., field id or column header).

In [ ]:
# Choose a numeric field to analyze
# Let's try to auto-detect numeric candidates
selected_record_set_id = main_record_set_id
df = dataframes[selected_record_set_id] if selected_record_set_id else None
if df is not None and not df.empty:
    numeric_candidates = df.select_dtypes(include=[int, float]).columns.tolist()
    print(f"Numeric fields found: {numeric_candidates}")
    # Choose first numeric field if possible
    numeric_field = numeric_candidates[0] if numeric_candidates else None
else:
    print("No rows available.")
    numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    print(f"\nFiltering on {numeric_field} > {threshold}")
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize selected numeric field in filtered dataframe
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to group by a categorical field
    # Choose first object (string) field not equal to numeric_field
    categorical_fields = [c for c in df.select_dtypes(include=['object']).columns if c != numeric_field]
    group_field = categorical_fields[0] if categorical_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric field found suitable for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field, or its relation to a group or categorical field, using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt

if df is not None and numeric_field:
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        # Boxplot by group, if group column has low cardinality
        n_unique = df[group_field].nunique()
        if n_unique > 1 and n_unique < 30:
            plt.figure(figsize=(10,6))
            df.boxplot(column=numeric_field, by=group_field, grid=False)
            plt.title(f"{numeric_field} by {group_field}")
            plt.suptitle("")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.show()
else:
    print("Unable to plot: no numeric field found.")

## 6. Conclusion

This notebook demonstrated how to:
- Load the FAIR^2 dataset using its Croissant schema URL,
- Identify record sets and their constituent fields by `@id`,
- Extract tabular data as pandas DataFrames using `mlcroissant`,
- Conduct basic filtering, normalization, grouping, and visualization.

**Key Findings:**
- Please review the outputs and plots above to interpret key statistical or biological trends in the dataset. As datasets can evolve, adapt field selections as needed based on the provided schema overview and column names.
For further details or more advanced ML tasks, follow up with additional cells for feature engineering, modeling, or export as required.